# 9.2 Sampling Complex Fields and Their Spectra

**When does an array represent the complex field we intended to calculate?**

We start with an already prescribed $U_{\mathrm{exit}}(x)$ at a fixed plane. Its exact Fourier coefficients are known, so we can measure what sampling and the FFT preserve or lose. No scattering or propagation calculation is needed for this experiment.

Our main optical problem is a periodic grating under plane-wave illumination. We represent **one period in 1D, or one unit cell in 2D**, with its repetition representing the ideal grating. The main experiments keep **$L=p=1000$ nm** and vary only the number of samples within that period.

By the end, you should be able to recover complex diffraction-order coefficients, identify aliasing, and connect the order index to its transverse wavevector. Window size, origin, and two-dimensional checks are optional at the end.

Review [Appendix 4](Appendix_4_How_to_Use_FFTs_in_Python.ipynb) for FS, FT, DFT, and Python FFT conventions. [9.0](9_0_From_a_Physical_Model_to_a_Numerical_Experiment.ipynb) places this lesson in the workflow; [9.1](9_1_Defining_the_Complex_Exit_Field.ipynb) explains what the exit field and its order representation mean. Computing $U_{\mathrm{exit}}$ from a mask and illumination belongs to 9.3.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'wave_optics').is_dir()), None)
if project_root is None:
    raise RuntimeError('Run this notebook from the project or one of its subdirectories.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Re-running the notebook releases only the explorer owned by this lesson.
previous = globals().pop('sampling_explorer', None)
if previous is not None:
    previous.close()

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
from wave_optics.amplitude_grating import field_fourier_coefficients
from wave_optics.field_sampling import periodic_field, sampling_experiment
from wave_optics.field_sampling_view import FieldSamplingExplorer, sampling_comparison_png

## 1. Define a controlled field for sampling

The [exit-field lesson, 9.1](9_1_Defining_the_Complex_Exit_Field.ipynb) distinguishes the spatial field, its order coefficients, and the periodic factor used for tilted illumination. Here we use normal incidence and prescribe a finite-harmonic field whose coefficients are known exactly. This isolates sampling error from physical-model error.

Here we prescribe the resulting coefficients directly, so that the exact answer is available for sampling checks. We have not yet selected a particular mask and illumination to generate them; that modeling step belongs to 9.3. Use a periodic scalar field with four nonzero complex coefficients:

$$U_{\mathrm{exit}}(x)=\sum_{m\in\{-3,0,2,5\}} C_m e^{i2\pi m x/p},\qquad p=1000\ \mathrm{nm}.$$

Each $C_m$ has an amplitude and a phase. Coordinates are in nm; field amplitudes use arbitrary units. This finite sum is deliberately smooth and exactly band limited. A sharp binary opening has infinitely many harmonics and requires a separate truncation choice.

We sample the **complex field**. Storing only $|U|$ or $|U|^2$ would discard phase information and would not determine the same Fourier coefficients. Intensity can also contain higher spatial frequencies than the field because it contains differences between field frequencies.

**Notation:** use $U_{\mathrm{exit}}$ at the exit plane, $U_{\mathrm{ref}}$ at a general reference plane, and $C_m$ (or $C_{mp}$) for field coefficients. [Shared convention](0_0_Foundations_of_Scalar_Optics.ipynb#exit-field-notation).

In [ ]:
p_nm = 1000.0
orders = np.array([-3, 0, 2, 5])
coefficients = np.array([
    0.25 * np.exp(0.6j),
    0.80 * np.exp(-0.3j),
    0.40 * np.exp(0.9j),
    0.20 * np.exp(-0.7j),
])

display(Markdown('| m | amplitude | phase (rad) |\n|---:|---:|---:|\n' + '\n'.join(
    f'| {m} | {abs(c):.2f} | {np.angle(c):+.2f} |'
    for m, c in zip(orders, coefficients))))

## 2. Sample one period and recover its coefficients

This section is a controlled representation check: evaluate the known Fourier sum on a spatial grid, then recover its coefficients from those samples.

$$
\underbrace{C_m}_{\text{known}}
\;\longrightarrow\;
\underbrace{U_{\mathrm{exit}}(x_j)}_{\text{spatial samples}}
\;\xrightarrow{\mathrm{FFT}}\;
\underbrace{\widehat C_m}_{\text{recovered}}.
$$

The recovered $\widehat C_m$ include the normalization and coordinate-origin correction below. Comparing them with the known $C_m$ establishes the correctly sampled baseline for the later aliasing experiment.

For **$N$ samples within one period**, set $L=p$:

$$x_j=x_{\mathrm{start}}+j\Delta x,\qquad \Delta x=L/N,\qquad j=0,\ldots,N-1.$$

The repeated endpoint is excluded: $x_{\mathrm{start}}+L$ is the next period's first point. Use `arange(N)` or `linspace(..., endpoint=False)`.

NumPy's default forward FFT returns $D_q=\sum_j U(x_j)e^{-i2\pi qj/N}$. To obtain coefficients multiplying $e^{i2\pi f_qx}$ in the physical coordinate,

$$\widetilde C_q=\frac{D_q}{N}e^{-i2\pi f_qx_{\mathrm{start}}},\qquad f_q=\frac{q}{L}.$$

`fftfreq` supplies signed $f_q$ in the same storage order as the FFT. With our one-period window $L=p$, the physical order $m$ occupies signed bin $q=m$. Negative bins use array indices `m % N`.

In [ ]:
N = 32
L_nm = p_nm
dx_nm = L_nm / N
x_start_nm = -L_nm / 2
x_nm = x_start_nm + np.arange(N) * dx_nm
U_exit = periodic_field(x_nm, p_nm, orders, coefficients)

fx = np.fft.fftfreq(N, d=dx_nm)  # cycles/nm, in raw FFT order
D = np.fft.fft(U_exit)
C_fft = (D / N) * np.exp(-2j * np.pi * fx * x_start_nm)
recovered = C_fft[orders % N]

# Reuse the existing one-period coefficient extractor from the grating lessons.
C_shared = field_fourier_coefficients(U_exit, x_nm, p_nm, orders)
display(Markdown('| m | known C_m | recovered C_m | absolute error |\n'
                 '|---:|---:|---:|---:|\n' + '\n'.join(
    f'| {m} | {c:.5f} | {r:.5f} | {abs(r-c):.2e} |'
    for m, c, r in zip(orders, coefficients, recovered))))
assert np.allclose(recovered, coefficients, atol=1e-12, rtol=0)
assert np.allclose(C_shared, recovered, atol=1e-12, rtol=0)

## 3. Put more sample points on the same field

Section 2 worked with **32 points over 1000 nm**. First change just one thing: use **64 points over the same 1000 nm**. Keep the physical field and its four coefficients fixed.

$$\underbrace{32\ \text{points}}_{\Delta x=31.25\ \mathrm{nm}}
\;\longrightarrow\;
\underbrace{64\ \text{points}}_{\Delta x=15.625\ \mathrm{nm}}
\qquad\text{over the same period.}$$

In the left panels, orange dots are the stored samples. We show $\operatorname{Re}U$ to make the dots easy to see; the FFT still uses the **full complex field**. On the right, black circles are the known coefficient magnitudes and orange crosses are the FFT results.

**Look for:** do the four nonzero coefficients change when the dots become closer together?

In [ ]:
display(Image(data=sampling_comparison_png(
    [(32, 1), (64, 1)], period_nm=p_nm, orders=orders, coefficients=coefficients)))

**What happened?** The second grid has twice as many dots, and both recover the same four coefficients. The 32-point grid already resolved this field. Extra samples extend the frequency range the FFT can represent; within the frequency range shown, the bin positions stay the same.

Here the **sample spacing** $\Delta x$ answers: how far apart are neighbouring dots in real space?

## 4. What if we use only eight points?

Keep the same one-period window. Compare the successful 32-point grid with **8 points over the same 1000 nm**. This time the dots are much farther apart.

The right-hand horizontal axis now labels the **order number $m$** directly. For eight samples, the FFT has bins $m=-4,-3,\ldots,3$; the grey regions lie outside its usual Nyquist interval. Black circles show the known field for reference, including orders the FFT cannot recover separately.

**Look for two things:** does the blue field still pass through the orange dots? Does it still follow the black field between those dots?

In [ ]:
display(Image(data=sampling_comparison_png(
    [(32, 1), (8, 1)], period_nm=p_nm, orders=orders,
    coefficients=coefficients, show_alias=True)))

**Read the lower-right panel as follows:**

| Marker | What it represents |
| --- | --- |
| Black circle at $m=-3$, height $0.25$ | Known $\lvert C_{-3}\rvert$ |
| Black circle at $m=5$, height $0.20$ | Known $\lvert C_5\rvert$; there is no separate eight-point FFT bin here |
| Red cross at $m=-3$, height about $0.36$ | Recovered $\lvert\widehat C_{-3}\rvert=\lvert C_{-3}+C_5\rvert$ |

The red cross contains **two complex contributions**, including their phases. The physical grating's $m=5$ harmonic stays at its original wavevector; the undersampled FFT misidentifies it as $m=-3$. This numerical confusion is **aliasing**. It explains why the blue field fits the stored dots but differs from the exact field between them.

Why these two orders? At $x_j=-p/2+jp/8$, their sampled phase factors are identical:

$$e^{i2\pi(5)x_j/p}=e^{i2\pi(-3)x_j/p}.$$

Their order numbers differ by eight, exactly the number of samples per period. The sampled array cannot distinguish the two contributions, so this FFT bin contains $\widehat C_{-3}=C_{-3}+C_5$. Complex coefficients add before we take their magnitudes.

### How many points would resolve this example?

The fastest term is $m=5$: five oscillations in 1000 nm, or one oscillation every **200 nm**. To distinguish the positive and negative frequencies in the usual FFT interval, put **more than two points per fastest oscillation**:

| Points per period | Point spacing | Points per fastest oscillation | Result for this field |
| --- | --- | --- | --- |
| 32 | 31.25 nm | 6.4 | resolved |
| 12 | 83.33 nm | 2.4 | resolved |
| 10 | 100 nm | 2 | positive and negative boundary frequencies coincide |
| 8 | 125 nm | 1.6 | aliased |

The usual name for this boundary is **Nyquist**. For the four harmonics used here, the condition is $|m|<N/2$, so $N>10$ samples within our one-period window. This checks representation of our prescribed field; a physical mask may require additional harmonics first.

In [ ]:
resolved_case = sampling_experiment(32, period_nm=p_nm,
                                    orders=orders, coefficients=coefficients)
aliased_case = sampling_experiment(8, period_nm=p_nm,
                                   orders=orders, coefficients=coefficients)
display(Markdown('| samples per period | FFT/inverse sample error | maximum fine-grid field error |\n'
                 '|---:|---:|---:|\n' + '\n'.join(
    f"| {d['samples_per_period']} | {d['roundtrip_error']:.2e} | {d['field_error']:.2e} |"
    for d in (resolved_case, aliased_case))))
aliased_coefficient = aliased_case['coefficients'][(-3) % 8]
expected_alias = coefficients[0] + coefficients[3]
print(f'FFT coefficient at m=-3: {aliased_coefficient:.6f}')
print(f'Known C_-3 + C_5:       {expected_alias:.6f}')
assert np.allclose(aliased_coefficient, expected_alias, atol=1e-12, rtol=0)

The first error column above is tiny for **both** grids. An FFT followed by an inverse FFT returns the stored array:

$$
\underbrace{U_{\mathrm{exit}}(x_j)}_{\text{stored samples}}
\;\xrightarrow{\mathrm{FFT}}\;
\underbrace{D_q}_{\text{DFT data}}
\;\xrightarrow{\text{inverse FFT}}\;
\underbrace{U_{\mathrm{exit}}(x_j)}_{\text{same samples}}.
$$

The second error column compares against the exact continuous field on a fine grid. That check exposes the eight-point grid's error between the stored samples.

## 5. Read the coefficients as diffraction orders

The one-period FFT gives a complex coefficient for each signed order $m$. Repeating the cell places these harmonics at the grating's discrete transverse wavevectors. At normal incidence,

$$f_{x,m}=\frac{m}{p},\qquad k_{x,m}=\frac{2\pi m}{p}=mG,\qquad G=\frac{2\pi}{p}.$$

For the $m=2$ term in our example,

$$\underbrace{m=2}_{\text{order}}
\;\xrightarrow{f_x=m/p}\;
\underbrace{f_x=0.002\ \mathrm{cycles/nm}}_{\text{two cycles per period}}
\;\xrightarrow{k_x=2\pi f_x}\;
\underbrace{k_x\approx0.01257\ \mathrm{rad/nm}}_{\text{transverse wavevector}}.$$

Its recovered coefficient is $C_2=0.40e^{i0.90}$: amplitude $0.40$ and phase $0.90$ rad at the exit plane. Increasing the sample count from 32 to 64 preserves this coefficient and its wavevector. With eight samples, some orders overlap in the sampled representation.

Thus **the period sets the order locations; the field sets their complex coefficients; the sample count controls whether we recover them correctly**. A wavelength and output medium are still needed to determine which orders propagate and their directions.

## What to carry into 9.3

For the main experiments, the physical period and the sampled window both stayed at **1000 nm**:

| Choice | Optical or numerical meaning |
| --- | --- |
| Period $p$ | Sets the physical harmonic spacing $G=2\pi/p$ |
| Samples per period $N$ | Sets the point spacing $\Delta x=p/N$ and which harmonics can be resolved |
| Recovered coefficient $C_m$ | Gives the amplitude and phase of the exit-field harmonic of order $m$ |

First sample one period finely enough, then check that further refinement preserves the complex coefficients. The eight-point example shows why matching the stored samples alone is insufficient.

**The main lesson ends here.** In 9.3 we will generate the exit field from a mask and illumination and apply the same one-cell coefficient checks. The following sections are optional reference material. Gaussian illumination and finite-window choices belong to 9.5.

## Optional A. Explore sampling within one period

The full explorer adds imaginary-part, amplitude, and phase panels. Keep **Periods in window = 1** and **Grid origin = Centred window** for these experiments:

1. Compare 32 and 64 samples within the same period.
2. Compare 32, 12, 10, and 8 samples to locate the onset of aliasing.

The prescribed physical field stays fixed. Spectral phases are hidden where coefficient magnitudes are negligible. The origin control is explained in Optional B; the multiple-period control is explained in Optional E.

In [ ]:
previous = globals().pop('sampling_explorer', None)
if previous is not None:
    previous.close()
sampling_explorer = FieldSamplingExplorer(
    period_nm=p_nm, orders=orders, coefficients=coefficients)
display(sampling_explorer.ui)

## Optional B. Move the sampling window

Imagine choosing a different starting point along the same periodic field. The field itself stays fixed; only the window moves. With enough samples, we should recover the same physical coefficients.

The origin correction in Section 2 makes that happen. Without it, `fft(U)/N` expresses phases relative to the first sample. Multiplying by $e^{-i2\pi f_qx_{\mathrm{start}}}$ expresses them in the original physical coordinate.

Read the table below across each row: moving the start changes the uncorrected result, while the corrected error remains tiny. `fftshift` rearranges bins; the phase correction above accounts for the coordinate origin.

In [ ]:
rows = []
for start_fraction in (-0.5, 0.0, 0.137):
    d = sampling_experiment(32, period_nm=p_nm, start_fraction=start_fraction,
                            orders=orders, coefficients=coefficients)
    uncorrected = d['raw_fft'][orders % d['N']] / d['N']
    corrected = d['coefficients'][orders % d['N']]
    rows.append(f"| {d['x_start_nm']:.0f} | {np.max(abs(uncorrected-coefficients)):.2e} | "
                f'{np.max(abs(corrected-coefficients)):.2e} |')
    assert np.allclose(corrected, coefficients, atol=1e-12, rtol=0)
display(Markdown('| start x (nm) | error without origin correction | error with correction |\n'
                 '|---:|---:|---:|\n' + '\n'.join(rows)))

## Optional C. Repeat the coefficient check in two dimensions

For a field periodic in both coordinates,

$$U(x,y)=\sum_{m,p}C_{mp}e^{i2\pi(mx/p_x+py/p_y)},\qquad
G_x=\frac{2\pi}{p_x},\quad G_y=\frac{2\pi}{p_y}.$$

With one period along each axis, coefficient $(m,p)$ appears at array index `[p % Ny, m % Nx]`: rows correspond to $y$, columns to $x$. Divide `fft2` by $N_xN_y$ and correct both coordinate origins. Each axis needs sufficient sampling: $|m|<N_x/2$ and $|p|<N_y/2$ for the harmonics retained here.

We prescribe another finite sum to verify the indexing independently of any scattering backend. Its transverse wavevector components are $(k_x,k_y)=(mG_x,pG_y)$.

For analytical rectangular-opening and nonseparable-mask coefficients, see the [unit-cell derivation below](#cell-coefficients-9-2).
The code retains `n` as the second-order identifier; it corresponds to mathematical $p$, not refractive index. Array ordering and FFT axes are unchanged.


:::{dropdown} Derivation: complex coefficients from one unit cell

(cell-coefficients-9-2)=

Use unit-amplitude normal incidence with zero incident phase at the origin for the coefficient examples below.

### Rectangular-opening coefficients

Prescribe $U_{\mathrm{exit}}(x,y)=t(x,y)U_i(x,y)$ at $z=0^-$. Orthogonality of the Fourier basis gives
$$t_{mp}=\frac1{p_xp_y}\int_{\rm cell}t(x,y)
 e^{-i2\pi(mx/p_x+py/p_y)}\,dx\,dy.$$
For a centered rectangular opening, the two integrals factor. Each integral, written with generic period $d$, is
$$\frac1d\int_{-Dd/2}^{Dd/2}e^{-i2\pi mx/d}\,dx=D\operatorname{sinc}(mD),$$
so
$$\boxed{t_{mp}=D_xD_y\operatorname{sinc}(mD_x)\operatorname{sinc}(pD_y)},\qquad
\operatorname{sinc}(u)=\frac{\sin\pi u}{\pi u}.$$
At the reference plane $z=0^-$, $C_{mp}=A_{\rm inc}t_{mp}$. We use $A_{\rm inc}=1$ with phase zero at the origin. This prescribed thin-mask calculation **does not solve material scattering**, Fresnel reflection, polarization, thickness, or coupling inside a physical grating.

An allowed location may have zero amplitude. Squared magnitudes $|t_{mp}|^2$ are coefficient weights, distinct from normal-flux efficiencies. For a scalar model whose flux is proportional to $\operatorname{Im}(U^*\partial_zU)$, a propagating transmitted order has relative normal flux $(-\operatorname{Re}k_{z,mp})/(-k_{z,\rm inc})\,|C_{mp}/A_{\rm inc}|^2$. Other field definitions can require impedance factors. A prescribed mask is not a solved energy balance; grazing and evanescent single orders carry no outgoing normal flux in this scalar convention.

Moving the opening by $\Delta\mathbf r$ at fixed coordinates multiplies its coefficient by $e^{-i(m\mathbf b_1+p\mathbf b_2)\cdot\Delta\mathbf r}$. Changing the coordinate origin instead also changes incident-reference phases: coefficient phases must always state their origin.

### Recover the same coefficients from unit-cell samples

Use endpoint-excluded samples $x_j/p_x=x_0+j/N_x$, $y_l/p_y=y_0+l/N_y$. The normalized FFT is
$$t^{\rm num}_{mp}=\frac{e^{-i2\pi(mx_0+py_0)}}{N_xN_y}
\sum_{j,l}t_{lj}e^{-i2\pi(mj/N_x+pl/N_y)}.$$
The origin factor matters. Midpoint sampling avoids jumps on sample points here; finite quadrature error remains. Never transform intensity in place of complex transmission.

Mathematical order labels are $(m,p)$; $p_x,p_y$ denote periods. Code identifiers `n`, `nn`, and `mn` retain their existing second-order and pair meanings.


### The one-dimensional limit

If $t$ is independent of $y$, integration over $y$ gives $\delta_{p0}$. Choosing $D_y=1$ recovers $t_{mp}=D_x\operatorname{sinc}(mD_x)\delta_{p0}$. All off-row locations still exist as candidate labels of this redundant two-period description, but they are unexcited. Conical incidence shifts the populated row to $k_y=k_{y,\rm inc}$.

### Nonseparable cells

Use two disjoint rectangles with different aspect ratios and centers in fractional cell coordinates. Their union is not a product $f(x)g(y)$: the available $y$ intervals depend on $x$. Its coefficient is the sum of two shifted rectangle coefficients. This also produces phases beyond 0 and $\pi$.


### Translation and the coefficient reference

A displaced opening changes phases at a fixed laboratory origin. If the new coordinates instead use an origin at $\mathbf r_0$, the full surface coefficient transforms as $C'_{mp}=C_{mp}e^{i\mathbf k_{\parallel,mp}\cdot\mathbf r_0}$; the mask coefficient uses only the reciprocal increment, and the incident amplitude gains its incident phase. Thus origin changes and physical translations are distinct operations.

At $z=0^-$ the coefficients above are normalized to $A_{\rm inc}=1$ with zero phase at the laboratory origin. At $z=-s$ they acquire $e^{-ik_{z,mp}s}$. An exact roll of the samples can be compared with the Fourier translation theorem and the analytic one-dimensional limit. The numerical coefficient errors are quadrature errors, not material-scattering errors.


:::


In [ ]:
px_nm, py_nm = 1000.0, 800.0
Nx, Ny = 32, 24
mn = np.array([[-2, 1], [0, 0], [1, -2], [3, 2]])
Cmn = np.array([0.20*np.exp(0.4j), 0.75, 0.35*np.exp(-0.6j), 0.25*np.exp(1.1j)])
x2_nm = -px_nm/2 + np.arange(Nx) * px_nm/Nx
y2_nm = -py_nm/2 + np.arange(Ny) * py_nm/Ny
X_nm, Y_nm = np.meshgrid(x2_nm, y2_nm)
U2 = sum(c * np.exp(2j*np.pi*(m*X_nm/px_nm + n*Y_nm/py_nm))
         for (m, n), c in zip(mn, Cmn))

fx2 = np.fft.fftfreq(Nx, d=px_nm/Nx)
fy2 = np.fft.fftfreq(Ny, d=py_nm/Ny)
FX, FY = np.meshgrid(fx2, fy2)
C2 = np.fft.fft2(U2) / (Nx*Ny)
C2 *= np.exp(-2j*np.pi*(FX*x2_nm[0] + FY*y2_nm[0]))
recovered_2d = C2[mn[:, 1] % Ny, mn[:, 0] % Nx]
expected_2d = np.zeros((Ny, Nx), dtype=complex)
expected_2d[mn[:, 1] % Ny, mn[:, 0] % Nx] = Cmn
print(f'Maximum error across all 2D coefficients: {np.max(abs(C2-expected_2d)):.2e}')
assert np.allclose(C2, expected_2d, atol=1e-12, rtol=0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
image = axes[0].pcolormesh(X_nm, Y_nm, abs(U2), shading='nearest', cmap='viridis')
axes[0].set(xlabel='x (nm)', ylabel='y (nm)', title='Prescribed 2D field amplitude', aspect='equal')
fig.colorbar(image, ax=axes[0], label='|U|')
axes[1].scatter(mn[:, 0], mn[:, 1], s=600*abs(Cmn), color='#2674af', label='Known')
axes[1].scatter(mn[:, 0], mn[:, 1], s=600*abs(recovered_2d), marker='x',
                color='#d46b19', label='FFT2 recovered')
axes[1].set(xlabel='Order m along x', ylabel='Order n along y',
            title='Nonzero coefficients: marker area scales with |Cmn|',
            xticks=np.arange(-3, 5), yticks=np.arange(-3, 4),
            xlim=(-3, 4), ylim=(-3, 3), aspect='equal')
axes[1].grid(alpha=0.2)
axes[1].legend(loc='lower left', fontsize=8, markerscale=0.65)
plt.show()
plt.close(fig)

## Optional D. Fourier-series coefficients versus a continuous FT

Section 2 divides the FFT by $N$ to recover **Fourier-series coefficients**. For an approximation to the continuous FT of a finite-window field, the scale is instead $\Delta x\,D_q$, with the same origin correction.

Since $\Delta x=L/N$, the two results differ by $L$ and have different units. [Appendix 4](Appendix_4_How_to_Use_FFTs_in_Python.ipynb) develops that distinction. All main experiments here use one complete period to isolate sampling; finite illuminated windows and leakage are developed in 9.5.

## Optional E. What if we sample more than one complete period?

One cell is sufficient for the periodic plane-wave problem above. This additional experiment compares computational windows; it introduces no new physical diffraction orders. Use 32 points per period. Now extend the sampled window from **1000 nm to 2000 nm**. This gives 64 points in total, with the original spacing of 31.25 nm. The grey curve outside the shaded region shows the same continuing periodic field.

$$\underbrace{1\ \text{period}}_{32\ \text{points}}
\;\longrightarrow\;
\underbrace{2\ \text{periods}}_{64\ \text{points}}
\qquad\text{with the same dot spacing.}$$

**Look for:** the four nonzero peaks stay at the same frequencies, but extra zero-valued FFT bins appear between the old bins. Follow the labelled $m=2$ peak.

In [ ]:
display(Image(data=sampling_comparison_png(
    [(32, 1), (32, 2)], period_nm=p_nm, orders=orders,
    coefficients=coefficients, highlight_order=2)))

**What happened?** The field still repeats every $p=1000$ nm. Its $m=2$ harmonic still has frequency $f_x=2/p=0.002$ cycles/nm and coefficient magnitude $0.40$. The longer window gives the FFT a finer frequency grid:

| Sampled window | FFT-bin spacing $1/L$ | Where the $m=2$ harmonic appears |
| --- | --- | --- |
| $L=1000$ nm | $0.001$ cycles/nm | signed bin $q=2$ |
| $L=2000$ nm | $0.0005$ cycles/nm | signed bin $q=4$ |

The **order number** $m$ identifies a harmonic of the physical repeat. The **bin number** $q$ identifies its place on the computational frequency grid. With two periods, $q=2m$; the new intermediate bins have zero coefficients for this exactly periodic test field.

### Now connect this picture to $G$ and $k_x$

Multiply spatial frequencies by $2\pi$ to express them as wavevector components. The physical harmonic spacing is $G=2\pi/p$. The FFT-bin spacing is $\Delta k_x=2\pi/L$.

Doubling $L$ halves the FFT-bin spacing. The physical period $p$, and therefore $G$, stay fixed. This is the same stationary-peak/more-bins picture above, expressed in rad/nm.

**Next, 9.3: Computing the Exit Field** will obtain the starting complex field from $t(x,y)U_i(x,y)$ or the scalar FMM backend, and compare different incident transverse wavevectors at a common exit plane.

Official references: [NumPy FFT definitions and normalization](#ref-9-2-numpy-fft) [\[1\]](#ref-9-2-numpy-fft), [signed frequency bins](#ref-9-2-numpy-fftfreq) [\[2\]](#ref-9-2-numpy-fftfreq), [FFT2 axes](#ref-9-2-numpy-fft2) [\[3\]](#ref-9-2-numpy-fft2).

[Previous: 9.1](9_1_Defining_the_Complex_Exit_Field.ipynb) | [Fourier/FFT appendix](Appendix_4_How_to_Use_FFTs_in_Python.ipynb)

## Worked-out examples

These optional scripts show the numerical implementation of the topics in this notebook.
Use the complete [scripts download](Downloads.md) to keep the plotting helper and
`wave_optics` package with the examples. Edit **INPUT PARAMETERS**, then run the file
or execute its `# %%` sections in order. Read **CHECK AND INTERPRET** with the plots.

### 1. Sample one period and recover its coefficients

**Script:** [9_2_01_fft_origin_correction.py](Wave_solver_py_scripts/scripts/chapter_9/9_2_01_fft_origin_correction.py)

Recover complex Fourier coefficients with the sampling-origin phase correction.

### 2. What if we use only eight points?

**Script:** [9_2_02_aliasing_collision.py](Wave_solver_py_scripts/scripts/chapter_9/9_2_02_aliasing_collision.py)

Demonstrate that a perfect FFT roundtrip can hide an aliased field.

### 3. Sample more than one complete period

**Script:** [9_2_03_multiple_period_windows.py](Wave_solver_py_scripts/scripts/chapter_9/9_2_03_multiple_period_windows.py)

Track FFT-bin changes when the window contains several periods.

### 4. Repeat the coefficient check in two dimensions

**Script:** [9_2_04_two_dimensional_fft_coefficients.py](Wave_solver_py_scripts/scripts/chapter_9/9_2_04_two_dimensional_fft_coefficients.py)

Recover a two-dimensional complex order lattice with FFT2.

(refs-9-2)=

## References

(ref-9-2-numpy-fft)=

[1] NumPy developers, “Discrete Fourier Transform: Implementation details and Normalization,” NumPy reference documentation. [Online]. Available: [source](https://numpy.org/doc/stable/reference/routines.fft.html).

(ref-9-2-numpy-fftfreq)=

[2] NumPy developers, “numpy.fft.fftfreq,” NumPy reference documentation. [Online]. Available: [source](https://numpy.org/doc/stable/reference/generated/numpy.fft.fftfreq.html).

(ref-9-2-numpy-fft2)=

[3] NumPy developers, “numpy.fft.fft2,” NumPy reference documentation. [Online]. Available: [source](https://numpy.org/doc/stable/reference/generated/numpy.fft.fft2.html).
